In [62]:
import os
from sklearn.metrics import accuracy_score, f1_score, jaccard_score, precision_score, recall_score
import matplotlib.pyplot as plt
import SinkTools2 as st
import tqdm
import torch
from osgeo import gdal
import numpy as np
import torch.nn.functional as F
from model import build_unet

threshold = 0.8
save_array = False
calc_each_samp = False
x_size, y_size = 256, 256

In [63]:
rasters_test_path = '/Users/nirgut/Documents/Master_Files/pipeline_testing/crop/DSM_tiles/normalized/test_data/'
masks_test_path = '/Users/nirgut/Documents/Master_Files/pipeline_testing/crop/Mask_tiles/test_masks/'
output_folder = '/Users/nirgut/Documents/Master_Files/pipeline_testing/crop/predictions/'
model_path = '/Users/nirgut/ML_Course/2nd_unet/files/no_overlap_epoch_53.pth'

In [64]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [65]:
def calculate_metrics(predictions, labels):
    # Convert predictions and labels to boolean arrays
    predictions = np.array(predictions) > threshold  # Assuming predictions are probabilities
    labels = np.array(labels) > threshold  # Assuming labels are binary (0 or 1)

    # True Positives (TP): Predicted positive and actually positive
    TP = np.sum(np.logical_and(predictions, labels))

    # False Positives (FP): Predicted positive but actually negative
    FP = np.sum(np.logical_and(predictions, np.logical_not(labels)))

    # True Negatives (TN): Predicted negative and actually negative
    TN = np.sum(np.logical_and(np.logical_not(predictions), np.logical_not(labels)))

    # False Negatives (FN): Predicted negative but actually positive
    FN = np.sum(np.logical_and(np.logical_not(predictions), labels))

    return TP, FP, TN, FN

In [66]:
def predict_mask(raster_file, checkpoint_path, device):
    """ Load model """
    model = build_unet()
    model = model.to(device)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()

    """ Reading raster """
    raster = gdal.Open(raster_file)
    raster = raster.ReadAsArray()

    """ Resize raster """
    new_height, new_width = 256, 256
    raster = raster.astype(np.float32)
    raster = torch.from_numpy(raster)
    raster = raster.unsqueeze(0)  # Add batch dimension
    raster = F.interpolate(raster.unsqueeze(0), size=(new_height, new_width), mode='bilinear', align_corners=False)

    """ Prediction """
    with torch.no_grad():
        pred_mask = model(raster)
        pred_mask = torch.sigmoid(pred_mask)
        pred_mask = pred_mask.cpu().numpy()
        pred_mask = np.squeeze(pred_mask, axis=0)  # Remove batch dimension
        # make a mask of 0s and 1s
        pred_mask = np.where(pred_mask >= threshold, 1, 0)


    return pred_mask


In [67]:
def model_full_test(rasters_folder, masks_folder, output_folder, model_path):
    # check if the output folder exists
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # load the model
    model = build_unet()
    model = model.to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    # load the rasters
    rasters = os.listdir(rasters_folder)
    rasters = [raster for raster in rasters if raster.endswith('.tif')]

    TP_total = 0
    FP_total = 0
    TN_total = 0
    FN_total = 0
    # use tqdm for progress bar

    for raster in tqdm.tqdm(rasters):
        # load the raster
        raster_path = os.path.join(rasters_folder, raster)
        mask_path = os.path.join(masks_folder, raster)
        # load the raster
        raster_array = gdal.Open(raster_path)
        geotrans = raster_array.GetGeoTransform()
        proj = raster_array.GetProjection()
        raster_array = raster_array.ReadAsArray()
        # load the mask
        mask = gdal.Open(mask_path)
        mask = mask.ReadAsArray()
        # if the shape of the maske is not the same as the predefined sizes, resize it

        # resize the mask
        # new_height, new_width = 256, 256
        mask = mask.astype(np.float32)
        # mask = torch.from_numpy(mask)
        if mask.shape != (x_size, y_size):
            new_height, new_width = 256, 256
            # mask = mask.astype(np.float32)
            mask = torch.from_numpy(mask)
            mask = mask.unsqueeze(0)  # Add batch dimension
            mask = F.interpolate(mask.unsqueeze(0), size=(new_height, new_width), mode='bilinear', align_corners=False)
        
        # mask = mask.unsqueeze(0)  # Add batch dimension
        # mask = F.interpolate(mask.unsqueeze(0), size=(new_height, new_width), mode='bilinear', align_corners=False)
        # predict single raster
        pred = predict_mask(raster_path, model_path, device)



        # calculate the metrics
        TP, FP, TN, FN = calculate_metrics(mask, pred)
        # calculate the metrics
        if calc_each_samp is True:
            sensitivity = TP / (TP + FN)
            specificity = TN / (TN + FP)
            # accuracy = (TP + TN) / (TP + FP + TN + FN)
            # precision = TP / (TP + FP)
            # f1 = 2 * (precision * sensitivity) / (precision + sensitivity)
            # jaccard = TP / (TP + FP + FN)
            print(f'\nraster name: {raster} \nSensitivity: {sensitivity}, specificity: {specificity}')
            # print(f'Sensitivity: {sensitivity}, specificity: {specificity}, accuracy: {accuracy}, precision: {precision}, f1: {f1}, jaccard: {jaccard}')
            # save the predicted mask

        if save_array is True:
            pred_path = os.path.join(output_folder, raster.split('.')[0] + '_pred.tif')
            # save array as raster according to the original raster using gdal not with st
            array2raster(raster_path, pred, pred_path, geotrans, proj)


        # update the total counts
        TP_total += TP
        FP_total += FP
        TN_total += TN
        FN_total += FN

    # calculate the total metrics
    sensitivity = TP_total / (TP_total + FN_total)
    specificity = TN_total / (TN_total + FP_total)
    accuracy = (TP_total + TN_total) / (TP_total + FP_total + TN_total + FN_total)
    precision = TP_total / (TP_total + FP_total)
    f1 = 2 * (precision * sensitivity) / (precision + sensitivity)
    jaccard = TP_total / (TP_total + FP_total + FN_total)
    print(f'Total sensitivity: {sensitivity}, total specificity: {specificity}, total accuracy: {accuracy}, total precision: {precision}, total f1: {f1}, total jaccard: {jaccard}')


In [68]:
def array2raster(raster_path, array, output_path, geotransform, projection):
    """
    Save a NumPy array as a raster file using GDAL.

    Parameters:
        raster_path (str): Path to the original raster file to use as a template.
        array (numpy.ndarray): NumPy array containing the data to be saved as a raster.
        output_path (str): Path to save the raster file.
        geotransform (tuple): Geotransform parameters (top left x, w-e pixel resolution, rotation, top left y, rotation, n-s pixel resolution).
        projection (str): Projection in WKT format.
    """
    # Open the original raster file
    original_raster = gdal.Open(raster_path)

    # Get the number of rows and columns from the original raster
    array = np.squeeze(array)
    # print(f'array shape: {array.shape}')
    rows, cols = array.shape

    # Create the output raster file
    driver = gdal.GetDriverByName('GTiff')
    out_raster = driver.Create(output_path, cols, rows, 1, gdal.GDT_Float32)

    # Write the array data to the output raster
    out_band = out_raster.GetRasterBand(1)
    out_band.WriteArray(array)

    # Set the geotransform and projection
    out_raster.SetGeoTransform(geotransform)
    out_raster.SetProjection(projection)

    # Close the output raster file
    out_band.FlushCache()

In [69]:
model_full_test(rasters_test_path, masks_test_path, output_folder, model_path)

100%|██████████| 10/10 [00:12<00:00,  1.29s/it]

Total sensitivity: 0.7556071152358855, total specificity: 0.992777423221207, total accuracy: 0.98809814453125, total precision: 0.678001387925052, total f1: 0.7147037307973664, total jaccard: 0.556061468412066
